- Up till this point, We were using Batch Gradient Descent, an optimization algorithm where the model calculates the gradients and updates its weights only after looking at the entire dataset.

- We generally dont use Batch Gradient descent due to below issues:
  - massive memory use
  - slow updates/convergence
  - trapped in local minima

- Solution is "Instead of loading entire data we should load batches and perform gradient descent on that batch only" - `(Mini Batch Gradient Descent)`

- Dataset & Dataloader helps to load data and itrate over it more efficiently and leave room for modifications how data need to be loaded for model training, apply transformations before training data etc.

# 📦 The `Dataset` and `DataLoader` Classes

`Dataset` and `DataLoader` are core abstractions in PyTorch that decouple how you define your data from how you efficiently iterate over it in training loops.

---

## 🏗️ The `Dataset` Class
The `Dataset` class acts as a blueprint. When you create a `Custom Dataset`, you decide how your data is loaded and returned. It requires you to define three specific methods:

* **`__init__()`**: Configures how your raw data files or data frames should be loaded and stored initially.
* **`__len__()`**: Returns the absolute total number of samples present inside your dataset.
* **`__getitem__(index)`**: Core method that fetches and returns a single data sample (and its corresponding label) at a specific integer index.

---

## 🔄 The `DataLoader` Class
The `DataLoader` wraps around your `Dataset` object and automates the process of batching, shuffling, and multi-process parallel loading.

### ⚙️ DataLoader Control Flow

- At the start of each epoch, the DataLoader (if shuffle=True)
shuffles indices(using a `sampler`)
- It divides the indices into chunks of `batch_size`.
- for each index in the chunk, data samples are fetched from
the Dataset object
- The samples are then collected and combined into a batch
(using `collate_fn`)
- The batch is returned to the main training loop



#### <i> <b>Note:</b> Dataset class knows where the data is in the memory, and the job of data loader class is to make bathes by requesting for the data inside the batches from dataset class</i>

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples = 10,     # number of samples
    n_features = 2,     # number of fetures
    n_informative = 2,  # number of informative features
    n_redundant = 0,    # number of redundant fetures
    n_classes = 2,      # number of classes
    random_state = 42   # for reproducibility
)

In [3]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [4]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [5]:
print(X.shape)
print(y.shape)

(10, 2)
(10,)


In [6]:
# convert the data to pytorch tensor
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.long)

In [7]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [8]:
y

tensor([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [9]:
# import Dataset and DataLoader classes
from torch.utils.data import Dataset, DataLoader

In [10]:
# Create CustomDataset class
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [11]:
# create object for CustomDataset class
dataset = CustomDataset(X, y)

In [12]:
len(dataset) # return rows of dataset

10

In [13]:
dataset[2] # third row of dataset

(tensor([-2.8954,  1.9769]), tensor(0))

In [14]:
# Create a DataLoader class Object (takes Dataset class object, batch size, either suffle or not)
dataloader = DataLoader(dataset, batch_size = 2, shuffle=True)

In [15]:
# dataloader is an itrable we can iterate on it to fetch the data.
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[-2.8954,  1.9769],
        [-0.5872, -1.9717]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------
tensor([[-0.7206, -0.9606],
        [ 1.7774,  1.5116]])
tensor([0, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-0.9382, -0.5430]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [-1.9629, -0.9923]])
tensor([1, 0])
--------------------------------------------------


#### Where to apply Data Transformations?
- Inside `__getitem__()` method in `CustomDataset()` class.

#### `Workers` in DataLoader
In PyTorch, the num_workers parameter in a DataLoader controls multi-process data loading.Setting workers allows PyTorch to load and prepare training data on your CPU in the background while your GPU is busy training the model.

## Applying Datasets and DataLoaders in Breast Cancer Code

In [16]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")

df.drop(columns=["id", "Unnamed: 32"], inplace = True)

"""### Train Test Split"""

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

"""### Scaling"""

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Need to encode labels
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

"""### Numpy arrays to Pytorch Tensors"""

X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()



In [17]:
print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([455, 30])
torch.Size([455])


In [21]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [22]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [23]:
train_dataset[10]

(tensor([-0.6615, -0.0375, -0.6973, -0.6475, -0.5822, -0.9648, -0.8900, -0.9395,
          0.4454, -0.2178, -0.8636,  0.3428, -0.8433, -0.6078, -0.3607, -0.9046,
         -0.6883, -0.6525,  0.3666, -0.7932, -0.7301,  0.1389, -0.7551, -0.6742,
         -0.4609, -0.9179, -0.9037, -0.8448,  1.0041, -0.8005]),
 tensor(0.))

In [24]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

In [25]:
"""### Defining the model"""

class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [26]:
"""Important Parameters"""

learning_rate = 0.1
epochs = 25

""" Adding Built-In Loss Function """
loss_function = nn.BCELoss()


In [27]:
"""### Training Pipeline"""

# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # The model.parameters() method in PyTorch retrieves an iterator over all the trainable parameters (weights and biases) in a model

# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features)

    # loss calculation
    loss = loss_function(y_pred, batch_labels.view(-1, 1))

    # its a good practice to reset gradients before backward pass
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameter update
    optimizer.step()

    # disable gradient tracking i.e. reset the gradients or avoid gradient accumulation (zero gradients)
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()

  # print the loss in each epoch
  print(f"Epoch: {epoch + 1}, loss: {loss.item()}")

Epoch: 1, loss: 0.18276353180408478
Epoch: 2, loss: 0.7533290982246399
Epoch: 3, loss: 0.1421387493610382
Epoch: 4, loss: 0.047386281192302704
Epoch: 5, loss: 0.032640490680933
Epoch: 6, loss: 0.017158864066004753
Epoch: 7, loss: 0.02059882879257202
Epoch: 8, loss: 0.02162066288292408
Epoch: 9, loss: 0.017876354977488518
Epoch: 10, loss: 0.17415665090084076
Epoch: 11, loss: 0.06809968501329422
Epoch: 12, loss: 0.01938583329319954
Epoch: 13, loss: 0.011039569973945618
Epoch: 14, loss: 0.06224662810564041
Epoch: 15, loss: 0.09116281569004059
Epoch: 16, loss: 0.025112653151154518
Epoch: 17, loss: 0.03385934233665466
Epoch: 18, loss: 0.005348194856196642
Epoch: 19, loss: 0.07708000391721725
Epoch: 20, loss: 0.014029862359166145
Epoch: 21, loss: 0.039907533675432205
Epoch: 22, loss: 0.005494479555636644
Epoch: 23, loss: 0.006851761136204004
Epoch: 24, loss: 0.04605179652571678
Epoch: 25, loss: 0.011988327838480473


In [30]:
# model evaluation using test loadre
model.eval() # set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # forward pass
    y_pred = model(batch_features)
    y_pred = (y_pred > 0.8).float() # convert probabilities to binary prediction

    # calculate accuracy for the current batch
    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

# calculate overall accuracy
overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f"Accuracy: {overall_accuracy:.4f}")

Accuracy: 0.9627
